In [1]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import (
    Adam,
    SGD,
    RMSprop
)
import keras_tuner as kt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    Flatten,
    Dense,
    Dropout,
    GlobalAveragePooling2D
)
from tensorflow.keras.applications import (
    MobileNetV2,
    EfficientNetB0,
    ResNet50,
    DenseNet121
)
import keras_tuner as kt
import pandas as pd
from sklearn.utils.class_weight import compute_class_weight

In [2]:
train_df = pd.read_csv("train.csv")
valid_df = pd.read_csv("validation.csv")
test_df = pd.read_csv("test.csv")

In [3]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
train_df["label"] = encoder.fit_transform(train_df["disease"])
valid_df["label"] = encoder.transform(valid_df["disease"])
test_df["label"] = encoder.transform(test_df["disease"])

In [4]:
IMG_HEIGHT = 224
IMG_WIDTH = 224
NUM_CLASSES = 7
INPUT_SHAPE = (224,224,3)
BATCH_SIZE = 32

In [5]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomContrast(0.2),
])

In [6]:
def load_image(path):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(
        image,
        channels=3
    )
    image = tf.image.resize(
        image,
        (224,224)
    )
    return image
def preprocess(path, label):
    image = load_image(path)
    image = tf.cast(image, tf.float32)
    image = data_augmentation(image)
    return image, label

In [7]:
train_ds = tf.data.Dataset.from_tensor_slices(
    (
        train_df["path"],
        train_df["label"]
    )
)
train_ds = train_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
train_ds = train_ds.shuffle(1000)
train_ds = train_ds.batch(BATCH_SIZE)
train_ds = train_ds.prefetch(
    tf.data.AUTOTUNE
)

In [8]:
valid_ds = tf.data.Dataset.from_tensor_slices(
    (
        valid_df["path"],
        valid_df["label"]
    )
)
valid_ds = valid_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
valid_ds = valid_ds.batch(BATCH_SIZE)
valid_ds = valid_ds.prefetch(
    tf.data.AUTOTUNE
)

In [9]:
test_ds = tf.data.Dataset.from_tensor_slices(
    (
        test_df["path"],
        test_df["label"]
    )
)
test_ds = test_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
test_ds = test_ds.batch(BATCH_SIZE)
test_ds = test_ds.prefetch(
    tf.data.AUTOTUNE
)

In [10]:
weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["label"]),
    y=train_df["label"]
)
print(weights)
class_weights = dict(zip(np.unique(train_df["label"]), weights))
print(class_weights)

[ 4.37305053  2.78174603  1.30224782 12.3633157   0.21338772  1.2855309
 10.11544012]
{np.int64(0): np.float64(4.37305053025577), np.int64(1): np.float64(2.7817460317460316), np.int64(2): np.float64(1.3022478172023035), np.int64(3): np.float64(12.36331569664903), np.int64(4): np.float64(0.21338772031292808), np.int64(5): np.float64(1.285530900421786), np.int64(6): np.float64(10.115440115440116)}


In [11]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

In [12]:
lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

In [14]:
base_model = EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=INPUT_SHAPE
)
base_model.trainable = False

In [15]:
efficientnet = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(256, activation="relu"),
    Dropout(0.5),
    Dense(NUM_CLASSES, activation="softmax")
])

In [16]:
efficientnet.compile(
    optimizer="SGD",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [18]:
history_eff = efficientnet.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=5,
    class_weight=class_weights,
    callbacks=[
        early_stop,
        lr_scheduler
    ]
)

Epoch 1/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 306s 1s/step - accuracy: 0.2892 - loss: 1.7833 - val_accuracy: 0.0140 - val_loss: 3.2913 - learning_rate: 0.0100
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 312s 1s/step - accuracy: 0.4020 - loss: 1.5610 - val_accuracy: 0.4953 - val_loss: 1.3949 - learning_rate: 0.0100
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 434s 2s/step - accuracy: 0.4621 - loss: 1.4073 - val_accuracy: 0.5732 - val_loss: 1.2212 - learning_rate: 0.0100
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 286s 1s/step - accuracy: 0.4919 - loss: 1.3123 - val_accuracy: 0.5226 - val_loss: 1.3093 - learning_rate: 0.0100
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 279s 1s/step - accuracy: 0.5223 - loss: 1.2590 - val_accuracy: 0.6045 - val_loss: 1.1183 - learning_rate: 0.0100
Restoring model weights from the end of the best epoch: 5.


In [19]:
train_loss, lr_train_acc_eff = efficientnet.evaluate(train_ds)
valid_loss, lr_valid_acc_eff = efficientnet.evaluate(valid_ds)
test_loss, lr_test_acc_eff = efficientnet.evaluate(test_ds)
print(lr_train_acc_eff)
print(lr_valid_acc_eff)
print(lr_test_acc_eff)

220/220 ━━━━━━━━━━━━━━━━━━━━ 322s 1s/step - accuracy: 0.6230 - loss: 1.0849
47/47 ━━━━━━━━━━━━━━━━━━━━ 44s 921ms/step - accuracy: 0.6059 - loss: 1.1245
47/47 ━━━━━━━━━━━━━━━━━━━━ 42s 874ms/step - accuracy: 0.5941 - loss: 1.1258
0.6229671835899353
0.6058588624000549
0.5941450595855713


In [20]:
eff_results = pd.DataFrame(columns=[
    "Model",
    "Train accuracy",
    "Test accuracy",
    "Valid accuracy"
])
eff_results.loc[len(eff_results)] = [
    "efficientnet using SGD",
    lr_train_acc_eff,
    lr_test_acc_eff,
    lr_valid_acc_eff
]
eff_results

,Model,Train accuracy,Test accuracy,Valid accuracy
0,efficientnet using SGD,0.622967,0.594145,0.605859


In [21]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

In [22]:
lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

In [23]:
base_model = EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=INPUT_SHAPE
)
base_model.trainable = False

In [24]:
efficientnet = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(256, activation="relu"),
    Dropout(0.5),
    Dense(NUM_CLASSES, activation="softmax")
])

In [25]:
efficientnet.compile(
    optimizer="RMSprop",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [26]:
history_eff = efficientnet.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=5,
    class_weight=class_weights,
    callbacks=[
        early_stop,
        lr_scheduler
    ]
)

Epoch 1/5


C:\Users\acer\miniconda3\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 240s 998ms/step - accuracy: 0.4294 - loss: 1.7091 - val_accuracy: 0.6565 - val_loss: 0.9495 - learning_rate: 0.0010
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 223s 996ms/step - accuracy: 0.5150 - loss: 1.4193 - val_accuracy: 0.6431 - val_loss: 0.9483 - learning_rate: 0.0010
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 225s 1s/step - accuracy: 0.5233 - loss: 1.3459 - val_accuracy: 0.6025 - val_loss: 1.0053 - learning_rate: 0.0010
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 224s 1s/step - accuracy: 0.5536 - loss: 1.3088 - val_accuracy: 0.6964 - val_loss: 0.7963 - learning_rate: 0.0010
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 222s 993ms/step - accuracy: 0.5725 - loss: 1.2631 - val_accuracy: 0.6731 - val_loss: 0.8102 - learning_rate: 0.0010
Restoring model weights from the end of the best epoch: 4.


In [27]:
train_loss, rm_train_acc_eff = efficientnet.evaluate(train_ds)
valid_loss, rm_valid_acc_eff = efficientnet.evaluate(valid_ds)
test_loss, rm_test_acc_eff = efficientnet.evaluate(test_ds)
print(rm_train_acc_eff)
print(rm_valid_acc_eff)
print(rm_test_acc_eff)

220/220 ━━━━━━━━━━━━━━━━━━━━ 179s 799ms/step - accuracy: 0.7127 - loss: 0.7531
47/47 ━━━━━━━━━━━━━━━━━━━━ 43s 916ms/step - accuracy: 0.7017 - loss: 0.7922
47/47 ━━━━━━━━━━━━━━━━━━━━ 40s 831ms/step - accuracy: 0.7026 - loss: 0.8218
0.7126961350440979
0.7017310261726379
0.7025948166847229


In [28]:
eff_results.loc[len(eff_results)] = [
    "efficientnet using RMSprop",
    rm_train_acc_eff,
    rm_test_acc_eff,
    rm_valid_acc_eff
]
eff_results

,Model,Train accuracy,Test accuracy,Valid accuracy
0,efficientnet using SGD,0.622967,0.594145,0.605859
1,efficientnet using RMSprop,0.712696,0.702595,0.701731


In [29]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

In [30]:
lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

In [31]:
base_model = EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=INPUT_SHAPE
)
base_model.trainable = False

In [32]:
efficientnet = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(256, activation="relu"),
    Dropout(0.5),
    Dense(NUM_CLASSES, activation="softmax")
])

In [33]:
efficientnet.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [34]:
history_eff = efficientnet.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=5,
    class_weight=class_weights,
    callbacks=[
        early_stop,
        lr_scheduler
    ]
)

Epoch 1/5


C:\Users\acer\miniconda3\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 239s 1000ms/step - accuracy: 0.4100 - loss: 1.6768 - val_accuracy: 0.5579 - val_loss: 1.2331 - learning_rate: 0.0010
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 234s 1s/step - accuracy: 0.4763 - loss: 1.3861 - val_accuracy: 0.6099 - val_loss: 1.0877 - learning_rate: 0.0010
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 220s 984ms/step - accuracy: 0.5063 - loss: 1.2973 - val_accuracy: 0.5033 - val_loss: 1.3290 - learning_rate: 0.0010
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 265s 1000ms/step - accuracy: 0.5251 - loss: 1.2187 - val_accuracy: 0.6072 - val_loss: 1.0216 - learning_rate: 0.0010
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 219s 977ms/step - accuracy: 0.5437 - loss: 1.1632 - val_accuracy: 0.4893 - val_loss: 1.2631 - learning_rate: 0.0010
Restoring model weights from the end of the best epoch: 4.


In [35]:
train_loss, adam_train_acc_eff = efficientnet.evaluate(train_ds)
valid_loss, adam_valid_acc_eff = efficientnet.evaluate(valid_ds)
test_loss, adam_test_acc_eff = efficientnet.evaluate(test_ds)
print(adam_train_acc_eff)
print(adam_valid_acc_eff)
print(adam_test_acc_eff)

220/220 ━━━━━━━━━━━━━━━━━━━━ 178s 795ms/step - accuracy: 0.6302 - loss: 0.9741
47/47 ━━━━━━━━━━━━━━━━━━━━ 38s 802ms/step - accuracy: 0.6205 - loss: 1.0186
47/47 ━━━━━━━━━━━━━━━━━━━━ 38s 811ms/step - accuracy: 0.5968 - loss: 1.0389
0.6302425265312195
0.6205059885978699
0.5968064069747925


In [36]:
eff_results.loc[len(eff_results)] = [
    "efficientnet using adam",
    adam_train_acc_eff,
    adam_test_acc_eff,
    adam_valid_acc_eff
]
eff_results

,Model,Train accuracy,Test accuracy,Valid accuracy
0,efficientnet using SGD,0.622967,0.594145,0.605859
1,efficientnet using RMSprop,0.712696,0.702595,0.701731
2,efficientnet using adam,0.630243,0.596806,0.620506


In [37]:
IMG_HEIGHT = 224
IMG_WIDTH = 224
NUM_CLASSES = 7
INPUT_SHAPE = (224,224,3)
BATCH_SIZE = 64

In [38]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomContrast(0.2),
])

In [39]:
def load_image(path):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(
        image,
        channels=3
    )
    image = tf.image.resize(
        image,
        (224,224)
    )
    return image
def preprocess(path, label):
    image = load_image(path)
    image = tf.cast(image, tf.float32)
    image = data_augmentation(image)
    return image, label

In [40]:
train_ds = tf.data.Dataset.from_tensor_slices(
    (
        train_df["path"],
        train_df["label"]
    )
)
train_ds = train_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
train_ds = train_ds.shuffle(1000)
train_ds = train_ds.batch(BATCH_SIZE)
train_ds = train_ds.prefetch(
    tf.data.AUTOTUNE
)

In [41]:
valid_ds = tf.data.Dataset.from_tensor_slices(
    (
        valid_df["path"],
        valid_df["label"]
    )
)
valid_ds = valid_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
valid_ds = valid_ds.batch(BATCH_SIZE)
valid_ds = valid_ds.prefetch(
    tf.data.AUTOTUNE
)

In [42]:
test_ds = tf.data.Dataset.from_tensor_slices(
    (
        test_df["path"],
        test_df["label"]
    )
)
test_ds = test_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
test_ds = test_ds.batch(BATCH_SIZE)
test_ds = test_ds.prefetch(
    tf.data.AUTOTUNE
)

In [43]:
weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["label"]),
    y=train_df["label"]
)
print(weights)
class_weights = dict(zip(np.unique(train_df["label"]), weights))
print(class_weights)

[ 4.37305053  2.78174603  1.30224782 12.3633157   0.21338772  1.2855309
 10.11544012]
{np.int64(0): np.float64(4.37305053025577), np.int64(1): np.float64(2.7817460317460316), np.int64(2): np.float64(1.3022478172023035), np.int64(3): np.float64(12.36331569664903), np.int64(4): np.float64(0.21338772031292808), np.int64(5): np.float64(1.285530900421786), np.int64(6): np.float64(10.115440115440116)}


In [44]:
base_model = EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=INPUT_SHAPE
)
base_model.trainable = False

In [45]:
efficientnet = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(256, activation="relu"),
    Dropout(0.5),
    Dense(NUM_CLASSES, activation="softmax")
])

In [46]:
efficientnet.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [47]:
batch_size = 64
history_eff = efficientnet.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=5,
    class_weight=class_weights,
    batch_size = batch_size
)

Epoch 1/5


C:\Users\acer\miniconda3\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


110/110 ━━━━━━━━━━━━━━━━━━━━ 241s 2s/step - accuracy: 0.4121 - loss: 1.6232 - val_accuracy: 0.5493 - val_loss: 1.2468
Epoch 2/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 20724s 190s/step - accuracy: 0.4957 - loss: 1.3292 - val_accuracy: 0.5679 - val_loss: 1.1264
Epoch 3/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 320s 3s/step - accuracy: 0.5198 - loss: 1.2386 - val_accuracy: 0.5965 - val_loss: 1.0451
Epoch 4/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 733s 7s/step - accuracy: 0.5466 - loss: 1.1721 - val_accuracy: 0.5979 - val_loss: 1.0755
Epoch 5/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 436s 4s/step - accuracy: 0.5519 - loss: 1.0973 - val_accuracy: 0.6192 - val_loss: 1.0202


In [48]:
train_loss, bt64_train_acc_eff = efficientnet.evaluate(train_ds)
valid_loss, bt64_valid_acc_eff = efficientnet.evaluate(valid_ds)
test_loss, bt64_test_acc_eff = efficientnet.evaluate(test_ds)
print(bt64_train_acc_eff)
print(bt64_valid_acc_eff)
print(bt64_test_acc_eff)

110/110 ━━━━━━━━━━━━━━━━━━━━ 308s 3s/step - accuracy: 0.6284 - loss: 0.9945
24/24 ━━━━━━━━━━━━━━━━━━━━ 52s 2s/step - accuracy: 0.6032 - loss: 1.0336
24/24 ━━━━━━━━━━━━━━━━━━━━ 49s 2s/step - accuracy: 0.5928 - loss: 1.0568
0.628387987613678
0.6031957268714905
0.5928143858909607


In [49]:
eff_results.loc[len(eff_results)] = [
    "efficientnet using batchsize  64",
    bt64_train_acc_eff,
    bt64_test_acc_eff,
    bt64_valid_acc_eff
]
eff_results

,Model,Train accuracy,Test accuracy,Valid accuracy
0,efficientnet using SGD,0.622967,0.594145,0.605859
1,efficientnet using RMSprop,0.712696,0.702595,0.701731
2,efficientnet using adam,0.630243,0.596806,0.620506
3,efficientnet using batchsize 64,0.628388,0.592814,0.603196


In [50]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

In [51]:
lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

In [52]:
base_model = efficientnet.layers[0]

base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

efficientnet.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_efficient_ft = efficientnet.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=5,
    class_weight=class_weights,
    callbacks=[
        early_stop,
        lr_scheduler
    ]
)

C:\Users\acer\miniconda3\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


Epoch 1/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 475s 3s/step - accuracy: 0.3569 - loss: 1.5771 - val_accuracy: 0.4514 - val_loss: 1.3523 - learning_rate: 1.0000e-05
Epoch 2/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 314s 3s/step - accuracy: 0.3842 - loss: 1.4225 - val_accuracy: 0.4554 - val_loss: 1.3527 - learning_rate: 1.0000e-05
Epoch 3/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 328s 3s/step - accuracy: 0.4184 - loss: 1.3649 - val_accuracy: 0.4807 - val_loss: 1.3092 - learning_rate: 1.0000e-05
Epoch 4/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 317s 3s/step - accuracy: 0.4405 - loss: 1.2962 - val_accuracy: 0.5047 - val_loss: 1.2716 - learning_rate: 1.0000e-05
Epoch 5/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 320s 3s/step - accuracy: 0.4578 - loss: 1.2548 - val_accuracy: 0.5379 - val_loss: 1.2221 - learning_rate: 1.0000e-05
Restoring model weights from the end of the best epoch: 5.


In [53]:
train_loss, fine_train_acc_eff = efficientnet.evaluate(train_ds)
valid_loss, fine_valid_acc_eff = efficientnet.evaluate(valid_ds)
test_loss, fine_test_acc_eff = efficientnet.evaluate(test_ds)
print(fine_train_acc_eff)
print(fine_valid_acc_eff)
print(fine_test_acc_eff)

110/110 ━━━━━━━━━━━━━━━━━━━━ 230s 2s/step - accuracy: 0.5317 - loss: 1.2072
24/24 ━━━━━━━━━━━━━━━━━━━━ 50s 2s/step - accuracy: 0.5260 - loss: 1.2346
24/24 ━━━━━━━━━━━━━━━━━━━━ 49s 2s/step - accuracy: 0.5223 - loss: 1.2666
0.531669020652771
0.5259653925895691
0.5222887396812439


In [54]:
eff_results.loc[len(eff_results)] = [
    "efficientnet using fine tunning",
    fine_train_acc_eff,
    fine_test_acc_eff,
    fine_valid_acc_eff
]
eff_results

,Model,Train accuracy,Test accuracy,Valid accuracy
0,efficientnet using SGD,0.622967,0.594145,0.605859
1,efficientnet using RMSprop,0.712696,0.702595,0.701731
2,efficientnet using adam,0.630243,0.596806,0.620506
3,efficientnet using batchsize 64,0.628388,0.592814,0.603196
4,efficientnet using fine tunning,0.531669,0.522289,0.525965


In [13]:
def build_efficientnet(hp):
    base = tf.keras.applications.EfficientNetB0(
        include_top=False,
        weights="imagenet",
        input_shape=INPUT_SHAPE
    )
    base.trainable=True
    model=tf.keras.Sequential([
        base,
        GlobalAveragePooling2D(),
        Dense(
            hp.Int(
                "units",
                64,
                128,
                64
            ),
            activation="relu"
        ),
        Dropout(
            hp.Float(
                "dropout",
                0.2,
                0.5,
                0.1
            )
        ),
        Dense(
            NUM_CLASSES,
            activation="softmax"
        )
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            hp.Float(
                "learning_rate",
                1e-5,
                1e-3,
                sampling="log"
            )
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

In [14]:
efficient_tuner = kt.RandomSearch(
    build_efficientnet,
    objective="val_accuracy",
    max_trials=3,
    directory="tuning",
    project_name="efficientnet"
)
efficient_tuner.search(
    train_ds,
    validation_data=valid_ds,
    epochs=5,
    class_weight=class_weights
)

Trial 3 Complete [00h 28m 03s]
val_accuracy: 0.7436751127243042

Best val_accuracy So Far: 0.7436751127243042
Total elapsed time: 01h 23m 32s


In [15]:
best_efficientnet = efficient_tuner.get_best_models(1)[0]

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/saving/saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 432 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [16]:
best_hps = efficient_tuner.get_best_hyperparameters(
    num_trials=1
)[0]
print(best_hps.values)

{'units': 128, 'dropout': 0.2, 'learning_rate': 0.0006590783805543402}


In [17]:
base_model = EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=INPUT_SHAPE
)
base_model.trainable = False

In [18]:
efficientnet = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(256, activation="relu"),
    Dropout(0.5),
    Dense(NUM_CLASSES, activation="softmax")
])

In [19]:
efficientnet.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [20]:
history_eff = efficientnet.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=5
)

Epoch 1/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 77s 327ms/step - accuracy: 0.6862 - loss: 0.8990 - val_accuracy: 0.7177 - val_loss: 0.7550
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 73s 324ms/step - accuracy: 0.7183 - loss: 0.7755 - val_accuracy: 0.7477 - val_loss: 0.6997
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 73s 323ms/step - accuracy: 0.7290 - loss: 0.7421 - val_accuracy: 0.7563 - val_loss: 0.6629
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 75s 334ms/step - accuracy: 0.7411 - loss: 0.7011 - val_accuracy: 0.7710 - val_loss: 0.6411
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 73s 324ms/step - accuracy: 0.7542 - loss: 0.6798 - val_accuracy: 0.7663 - val_loss: 0.6147


In [21]:
train_loss, hype_train_acc_eff = efficientnet.evaluate(train_ds)
valid_loss, hype_valid_acc_eff = efficientnet.evaluate(valid_ds)
test_loss, hype_test_acc_eff = efficientnet.evaluate(test_ds)
print(hype_train_acc_eff)
print(hype_valid_acc_eff)
print(hype_test_acc_eff)

220/220 ━━━━━━━━━━━━━━━━━━━━ 59s 262ms/step - accuracy: 0.7775 - loss: 0.5926
47/47 ━━━━━━━━━━━━━━━━━━━━ 13s 269ms/step - accuracy: 0.7617 - loss: 0.6363
47/47 ━━━━━━━━━━━━━━━━━━━━ 13s 267ms/step - accuracy: 0.7538 - loss: 0.6710
0.7774607539176941
0.7616511583328247
0.7538256645202637


In [ ]:
eff_results.loc[len(eff_results)] = [
    "efficientnet using hyperparameter",
    hype_train_acc_eff,
    hype_test_acc_eff,
    hype_valid_acc_eff
]
eff_results

In [ ]:
eff_results.to_csv("efficientnet_comparison.csv",index=False)

In [26]:
best_efficientnet.save("cnn_efficientnet_phase5.keras")

In [27]:
efficientnet.save("cnn_efficientnet_bestmodel.keras")